# Bronze: Dimensões

**Objetivo:** copiar fielmente as tabelas Raw das 3 dimensões (Produtos, Lojas, Representantes) para a camada Bronze, sem nenhuma transformação de conteúdo — apenas adicionando metadados de controle. A Bronze é a "fonte da verdade" bruta e imutável, servindo de base para todas as transformações que ocorrerão exclusivamente na Silver.

**Fonte:** tabelas Delta `raw.produtos`, `raw.lojas`, `raw.representantes`.

**Destino:** tabelas Delta `bronze.produtos`, `bronze.lojas`, `bronze.representantes`.

**Modo de execução:** streaming em batch (`trigger(availableNow=True)`), lendo incrementalmente da Raw via Structured Streaming sobre tabela Delta (não mais Autoloader/`cloudFiles`, já que a fonte agora é uma tabela Delta, não arquivos).

**Metadados adicionados:** `data_ingestao_bronze` (timestamp de quando o dado chegou na Bronze), mantendo também o `data_ingestao` original vindo da Raw.

**Observação de arquitetura:** nenhum casting de tipo, padronização de texto ou cálculo é realizado aqui — os preços continuam como texto, os nomes mantêm qualquer inconsistência da origem. Essas responsabilidades pertencem exclusivamente à Silver.

In [0]:
# Imports e configuração do schema

from pyspark.sql.functions import current_timestamp

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.bronze")

print("Schema 'bronze' verificado/criado com sucesso.")

In [0]:
# Bronze produtos

caminho_checkpoint_bronze_produtos = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/bronze/produtos"

df_stream_bronze_produtos = (
    spark.readStream
    .format("delta")
    .table("poc_latam_food.raw.produtos")
    .withColumn("data_ingestao_bronze", current_timestamp())
)

(
    df_stream_bronze_produtos
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_bronze_produtos)
    .trigger(availableNow=True)
    .toTable("poc_latam_food.bronze.produtos")
)

print("Bronze de Produtos concluído.")

In [0]:
# Bronze lojas

caminho_checkpoint_bronze_lojas = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/bronze/lojas"

df_stream_bronze_lojas = (
    spark.readStream
    .format("delta")
    .table("poc_latam_food.raw.lojas")
    .withColumn("data_ingestao_bronze", current_timestamp())
)

(
    df_stream_bronze_lojas
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_bronze_lojas)
    .trigger(availableNow=True)
    .toTable("poc_latam_food.bronze.lojas")
)

print("Bronze de Lojas concluído.")

In [0]:
# Bronze representantes

caminho_checkpoint_bronze_representantes = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/bronze/representantes"

df_stream_bronze_representantes = (
    spark.readStream
    .format("delta")
    .table("poc_latam_food.raw.representantes")
    .withColumn("data_ingestao_bronze", current_timestamp())
)

(
    df_stream_bronze_representantes
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_bronze_representantes)
    .trigger(availableNow=True)
    .toTable("poc_latam_food.bronze.representantes")
)

print("Bronze de Representantes concluído.")

In [0]:
# Validação visual - conferência das tabelas Bronze

print("=== bronze.produtos ===")
spark.table("poc_latam_food.bronze.produtos").display()

print("=== bronze.lojas ===")
spark.table("poc_latam_food.bronze.lojas").display()

print("=== bronze.representantes ===")
spark.table("poc_latam_food.bronze.representantes").display()

print("Contagens:")
print(f"Produtos: {spark.table('poc_latam_food.bronze.produtos').count()}")
print(f"Lojas: {spark.table('poc_latam_food.bronze.lojas').count()}")
print(f"Representantes: {spark.table('poc_latam_food.bronze.representantes').count()}")